# analisi STORM PRISM insieme

In [ ]:
import sys
sys.path.append('sources')
from sources.etl.cpi import analize_cpi
from sources.sampler import sample_expected_impact

from etl.cpi import load_cpi_file
process_name = "choice" #"" #choice-task-init-reverse

cpi_dict = load_cpi_file(process_name)
thresholds = sample_expected_impact(cpi_dict)
if cpi_dict:
	spin_model = analize_cpi(cpi_dict)
# prism_model = spin_model.generate_prism_model()
output_file =  "models/" + process_name + '.nm'
# with open(output_file, 'w') as f:
# 	f.write(prism_model)


In [ ]:
import os
from typing import Dict, Any
from sources.analysis import *
from sources.bounds import generate_multi_rewards_requirement
import subprocess
import os
from typing import Optional
import graphviz
import json
import re
from datetime import datetime

from sources.analysis import parse_line_value

def run_prism_analysis(process_name, pctl_path, create_mdp=False, verbose=False) -> Dict[str, Any]:
    """
    Runs PRISM analysis on a model file and saves results.
    
    Args:
        process_name (str): Name of the process (without extension)
        
    Returns:
        dict: Analysis information including modules, variables, and timing
    """
    # Define paths
    os.makedirs(os.path.join("models"), exist_ok=True)
    model_path = os.path.join("models", f"{process_name}.nm")
    dot_path = os.path.join("models", f"{process_name}.dot")
    info_path = os.path.join("models", f"{process_name}.info")
    cpi_path = os.path.join("CPIs", f"{process_name}.cpi")
    states_path = os.path.join("models", f"{process_name}_states.csv")
    trans_path = os.path.join("models", f"{process_name}_trans.tra")
    # https://www.prismmodelchecker.org/manual/Appendices/ExplicitModelFiles#tra
    # https://www.prismmodelchecker.org/manual/RunningPRISM/ExportingTheModel#formats


    # Read CPI file to get task impacts
    with open(cpi_path, 'r') as f:
        cpi_data = json.load(f)


    # Run PRISM command
    cmd = [
            os.path.abspath(PRISM_PATH) if PRISM_PATH else "prism",
            "-cuddmaxmem",
            "10g",
            "-javamaxmem",
            "2g",
            os.path.abspath(model_path),
            os.path.abspath(pctl_path)
        ]

    if create_mdp:
        cmd += ["-exportstates", os.path.abspath(states_path),
                "-exporttrans", os.path.abspath(trans_path)]

    cmd.append("-verbose")


    try:
        result = subprocess.run(cmd, 
                              capture_output=True, 
                              text=True, 
                              check=True)
        
         # Parse PRISM output
        prism_output = result.stdout
        model_info: Dict[str, Any] = {}
        timings: Dict[str, Optional[float]] = {}
        states_info: Dict[str, Optional[int]] = {}
        result_value: Optional[bool] = None
        warnings: list[str] = []

        if verbose:
            print(f'Prism output:\n{prism_output}')
        
        for line in prism_output.split('\n'):
            line = line.strip()
            
            # Version and basic info
            if value := parse_line_value(line, 'Version:'): 
                model_info['version'] = value
            elif value := parse_line_value(line, 'Type:'): 
                model_info['type'] = value
            elif value := parse_line_value(line, 'Modules:'):
                model_info['modules'] = value.split()
            elif value := parse_line_value(line, 'Variables:'):
                model_info['variables'] = value.split()
                
            # Timing information
            elif 'Time for model construction:' in line:
                if value := parse_line_value(line, 'Time for model construction:'):
                    timings['model_construction'] = safe_float_conversion(value)
            elif 'Time for model checking:' in line:
                if value := parse_line_value(line, 'Time for model checking:'):
                    timings['model_checking'] = safe_float_conversion(value)
                
            # States information
            elif line.startswith('States:'):
                total, initial = parse_states_line(line)
                states_info['total'] = total
                states_info['initial'] = initial
            elif value := parse_line_value(line, 'Transitions:'):
                states_info['transitions'] = safe_int_conversion(value)
            elif value := parse_line_value(line, 'Choices:'):
                states_info['choices'] = safe_int_conversion(value)
                
            # Result
            elif value := parse_line_value(line, 'Result:'):
                result_value = value.lower() == 'true'
                
            # Warnings
            elif line.startswith('Warning:'):
                warnings.append(line.split('Warning:', 1)[1].strip())
        
        # Compile complete results
        return {
            'command': ' '.join(cmd),
            'prism_output': prism_output,
            'model_info': model_info,
            'timings': timings,
            'states_info': states_info,
            'result': result_value,
            'warnings': warnings,
            'return_code': result.returncode,
            'error_output': result.stderr if result.stderr else None
        }
        
    except subprocess.CalledProcessError as e:
        return {
            'command': ' '.join(cmd),
            'error': str(e),
            'prism_output': e.output if hasattr(e, 'output') else None,
            'return_code': e.returncode,
            'error_output': e.stderr if hasattr(e, 'stderr') else None,
            'result': None,
            'model_info': {},
            'timings': {},
            'states_info': {},
            'warnings': [],
        }
    except Exception as e:
        return {
            'command': ' '.join(cmd),
            'error': str(e),
            'return_code': -1,
            'result': None,
            'model_info': {},
            'timings': {},
            'states_info': {},
            'warnings': [],
        }
import time
import stormpy


def run_storm_analysis(output_file: str, property_str: str = 'multi(R{"impact_1"}<=89 [C])', verbose=False):   
    """
    Runs STORM analysis on a model file and saves results.
    Args:
        output_file (str): Path to the model file .nm
        property_str (str): Property to be checked
        verbose (bool): Whether to print detailed logs
    Returns:
        dict: Analysis information including result and timing
    """    
    if verbose:
        print("\n=== Running STORM Analysis ===")
    storm_start_time = time.time()
    
    try:
        if verbose:
            print(f"Loading STORM model from: {output_file}")
        # Parse PRISM program
        parse_start = time.time()
        prism_program = stormpy.parse_prism_program(output_file)
        parse_time = time.time() - parse_start
        
        # Build model
        build_start = time.time()
        model = stormpy.build_model(prism_program)
        build_time = time.time() - build_start
        if verbose:
            print(f"STORM model built successfully: {model}")
        
        # Parse properties
        prop_parse_start = time.time()
        properties = stormpy.parse_properties(property_str, prism_program)
        prop_parse_time = time.time() - prop_parse_start
        if verbose:
            print(f"STORM properties parsed successfully: {properties}")
        
        # Model checking
        checking_start = time.time()
        result = stormpy.model_checking(model, properties[0])
        checking_time = time.time() - checking_start
        
        storm_total_time = time.time() - storm_start_time
        
        if verbose:
            print(f"STORM analysis result: {result}")
            print(f"\nSTORM Total Time: {storm_total_time:.3f}s")
            print(f"  - Parse program time: {parse_time:.3f}s")
            print(f"  - Build model time: {build_time:.3f}s")
            print(f"  - Parse properties time: {prop_parse_time:.3f}s")
            print(f"  - Model checking time: {checking_time:.3f}s")
        
        return {
            'result': result,
            'timings': {
                'total_elapsed': storm_total_time,
                'parse_program': parse_time,
                'build_model': build_time,
                'parse_properties': prop_parse_time,
                'model_checking': checking_time
            },
            'model_info': {
                'states': model.nr_states,
                'transitions': model.nr_transitions,
                'choices': model.nr_choices
            },
            'property': property_str,
            'model': model
        }
        
    except Exception as e:
        storm_total_time = time.time() - storm_start_time
        print(f"STORM error: {str(e)}")
        return {
            'error': str(e),
            'timings': {'total_elapsed': storm_total_time}
        }
def analyze_bounds(model_name: str, thresholds: Dict[str, float], prism=True, storm=True, verbose = False) -> Dict[str, Any]:
    """
    Analyze a model against multi-reward bounds.
    
    Args:
        model_name: Name of the model file (without extension)
        thresholds: Dictionary mapping impact names to threshold values
        
    Returns:
        Analysis results including full PRISM analysis information
    """
    # Ensure models directory exists
    os.makedirs('models', exist_ok=True)
    results = {}
    # Define paths
    model_path = os.path.join('models', f'{model_name}.nm')
    pctl_path = os.path.join('models', f'{model_name}.pctl')
    # Generate and write PCTL property
    property_str = generate_multi_rewards_requirement(thresholds)
    print("Generated property:", property_str)
    if verbose:
        with open(model_path, 'r') as f:
            print(f.read())
    try:
        with open(pctl_path, 'w') as f:
            f.write(property_str)
    except IOError as e:
        results['ptlc']={
            'error': f"Failed to write PCTL file: {str(e)}",
            'return_code': -1,
            'result': None,
            'model_info': {},
            'timings': {},
            'states_info': {},
            'warnings': [],
            'property': property_str
        }
        return results
    if verbose:
        print("pctl_path: ", pctl_path)
        print("model_path: ", model_path)
    if prism:
        results['prism'] = run_prism_analysis(model_name, pctl_path=pctl_path, create_mdp=True, verbose=verbose)
    if storm:
        results['storm'] = run_storm_analysis(model_path, property_str=property_str, verbose=verbose) 
    return results

In [ ]:
print("\n=== Analyzing Bounds ===")
print("process_name:", process_name)
print("thresholds:", thresholds)
print("==============================")
results = analyze_bounds(process_name, thresholds)

In [ ]:
print("\n=== Analyzing Bounds Results ===")
for tool, res in results.items():
    print(f"\n--- Results from {tool.upper()} ---")
    if 'error' in res:
        print(f"Error: {res['error']}")
    else:
        if 'result' in res:
            print(f"Result: {res['result']}")
        if 'timings' in res and res['timings']:
            print("Timings:")
            for timing, duration in res['timings'].items():
                if isinstance(duration, (int, float)):
                    print(f"  {timing}: {duration:.3f}s")
                else:
                    print(f"  {timing}: {duration}")
        if 'model_info' in res and res['model_info']:
            print("Model Info:")
            for key, value in res['model_info'].items():
                print(f"  {key}: {value}")

In [ ]:
print("\n=== Model Comparison ===")

if 'prism' in results and 'storm' in results:
    prism_res = results['prism']
    storm_res = results['storm']
    
    # Get PRISM model info
    prism_states = prism_res.get('states_info', {}).get('total')
    prism_transitions = prism_res.get('states_info', {}).get('transitions')
    prism_choices = prism_res.get('states_info', {}).get('choices')
    
    # Get STORM model info
    storm_states = storm_res.get('model_info', {}).get('states')
    storm_transitions = storm_res.get('model_info', {}).get('transitions')
    storm_choices = storm_res.get('model_info', {}).get('choices')
    
    print("\nPRISM Model:")
    print(f"  States: {prism_states}")
    print(f"  Transitions: {prism_transitions}")
    print(f"  Choices: {prism_choices}")
    
    print("\nSTORM Model:")
    print(f"  States: {storm_states}")
    print(f"  Transitions: {storm_transitions}")
    print(f"  Choices: {storm_choices}")
    
    print("\nComparison:")
    states_match = prism_states == storm_states
    transitions_match = prism_transitions == storm_transitions
    choices_match = prism_choices == storm_choices
    
    print(f"  States match: {states_match} [{'MATCH' if states_match else 'DIFF'}]")
    print(f"  Transitions match: {transitions_match} [{'MATCH' if transitions_match else 'DIFF'}]")
    print(f"  Choices match: {choices_match} [{'MATCH' if choices_match else 'DIFF'}]")
    
    if states_match and transitions_match and choices_match:
        print("\n[OK] Models are IDENTICAL")
    else:
        print("\n[ERROR] Models are DIFFERENT")
        if not states_match:
            print(f"  State difference: PRISM={prism_states}, STORM={storm_states}")
        if not transitions_match:
            print(f"  Transition difference: PRISM={prism_transitions}, STORM={storm_transitions}")
        if not choices_match:
            print(f"  Choice difference: PRISM={prism_choices}, STORM={storm_choices}")
else:
    print("Cannot compare: Both PRISM and STORM results are required")

In [ ]:
print("\n=== Detailed Model Structure Comparison ===")

if 'prism' in results and 'storm' in results and 'model' in results['storm']:
    storm_model = results['storm']['model']
    
    # Read PRISM exported files
    states_path = os.path.join("models", f"{process_name}_states.csv")
    trans_path = os.path.join("models", f"{process_name}_trans.tra")
    
    print(f"\nReading PRISM exported files:")
    print(f"  States file: {states_path}")
    print(f"  Transitions file: {trans_path}")
    
    # Parse PRISM states file
    prism_states = {}
    if os.path.exists(states_path):
        with open(states_path, 'r') as f:
            lines = f.readlines()
            print(f"\n  Found {len(lines)} lines in states file")
            for line in lines[:5]:  # Show first few states
                print(f"    {line.strip()}")
            if len(lines) > 5:
                print(f"    ... ({len(lines) - 5} more states)")
    else:
        print(f"  [WARNING] States file not found!")
    
    # Parse PRISM transitions file
    prism_transitions = []
    if os.path.exists(trans_path):
        with open(trans_path, 'r') as f:
            lines = f.readlines()
            print(f"\n  Found {len(lines)} lines in transitions file")
            # First line is header
            if lines:
                print(f"    Header: {lines[0].strip()}")
            for line in lines[1:6]:  # Show first few transitions
                print(f"    {line.strip()}")
            if len(lines) > 6:
                print(f"    ... ({len(lines) - 6} more transitions)")
    else:
        print(f"  [WARNING] Transitions file not found!")
    
    print("\n--- STORM Model Details ---")
    print(f"Model type: {storm_model.model_type}")
    print(f"States: {storm_model.nr_states}")
    print(f"Transitions: {storm_model.nr_transitions}")
    print(f"Choices: {storm_model.nr_choices}")
    
    # Analyze STORM model structure
    print("\nSTORM State Analysis (first 10 states):")
    for state_id in range(min(10, storm_model.nr_states)):
        if storm_model.has_choice_labeling():
            labels = storm_model.choice_labeling.get_labels_of_state(state_id)
        else:
            labels = []
        
        # Get transitions from this state
        if state_id < storm_model.nr_states:
            row_group_start = storm_model.transition_matrix.get_row_group_start(state_id)
            row_group_end = storm_model.transition_matrix.get_row_group_end(state_id)
            num_choices = row_group_end - row_group_start
            
            print(f"  State {state_id}: {num_choices} choice(s)", end="")
            if labels:
                print(f" [labels: {labels}]", end="")
            print()
            
            # Show transitions for first few choices
            for choice_idx in range(row_group_start, min(row_group_start + 2, row_group_end)):
                for entry in storm_model.transition_matrix.get_row(choice_idx):
                    print(f"    -> State {entry.column} (prob: {entry.value()})")
    
    if storm_model.nr_states > 10:
        print(f"  ... ({storm_model.nr_states - 10} more states)")
    
    print("\n--- Comparing Model Structures ---")
    
    # Check if files were exported successfully
    if os.path.exists(states_path) and os.path.exists(trans_path):
        # Count PRISM states and transitions
        with open(states_path, 'r') as f:
            prism_state_count = len(f.readlines())
        
        with open(trans_path, 'r') as f:
            prism_trans_count = len(f.readlines()) - 1  # Subtract header
        
        print(f"\nPRISM exported:")
        print(f"  States: {prism_state_count}")
        print(f"  Transitions: {prism_trans_count}")
        
        print(f"\nSTORM model:")
        print(f"  States: {storm_model.nr_states}")
        print(f"  Transitions: {storm_model.nr_transitions}")
        
        # Compare
        if prism_state_count == storm_model.nr_states and prism_trans_count == storm_model.nr_transitions:
            print("\n[OK] PRISM and STORM models have matching structure!")
        else:
            print("\n[ERROR] Models have DIFFERENT structures:")
            if prism_state_count != storm_model.nr_states:
                print(f"  States: PRISM={prism_state_count}, STORM={storm_model.nr_states}")
            if prism_trans_count != storm_model.nr_transitions:
                print(f"  Transitions: PRISM={prism_trans_count}, STORM={storm_model.nr_transitions}")
    else:
        print("\n[WARNING] Cannot perform detailed comparison: PRISM export files not found")
        print("  Make sure create_mdp=True was used in run_prism_analysis")

else:
    print("Cannot perform detailed comparison: STORM model not available")